# Session 4 — FeatureBuilder

Cleaning gave me a gap-free daily AQI series per city. Now I turn that raw series
into a model-ready table. The job of this session is to build features that
describe everything known up to and including day T, paired with a target that is
day T+1's AQI — so the model learns to forecast tomorrow from today and the recent
past, never from tomorrow itself.

## 0. Load the clean frame and lock the contract

Before I build a single feature I reload the cleaned city-day frame, re-sort it by
(city_id, date), and assert the dates climb in order inside every city. Every lag
and rolling feature I build next leans entirely on this ordering — if it's wrong,
the leakage guarantees silently break, with no error to warn me. So I prove it
once, here, with a hard assert.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# make the src package importable from inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config

# --- load the cleaned city-day frame ---
df = pd.read_parquet(config.CLEAN_CITY_DAY_PATH)

# --- re-sort defensively: features depend entirely on this order ---
df = df.sort_values([config.CITY_ID_COL, config.DATE_COL]).reset_index(drop=True)

# --- prove dates climb in order inside EVERY city (the leakage guard) ---
dates_ok = df.groupby(config.CITY_ID_COL)[config.DATE_COL].apply(
    lambda s: s.is_monotonic_increasing
).all()
assert dates_ok, "dates are not strictly ordered within every city"

print("shape       :", df.shape)
print("cities      :", df[config.CITY_ID_COL].nunique())
print("date range  :", df[config.DATE_COL].min(), "->", df[config.DATE_COL].max())
print("columns     :", list(df.columns))
df.head()

shape       : (224808, 15)
cities      : 141
date range  : 2018-01-01 00:00:00 -> 2023-03-31 00:00:00
columns     : ['city_id', 'city', 'state', 'date', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'aqi', 'aqi_bucket']


,city_id,city,state,date,pm25,pm10,no,no2,nox,nh3,so2,co,o3,aqi,aqi_bucket
0,"Agartala, Tripura",Agartala,Tripura,2020-11-08,53.016667,68.795000,1.323333,6.743333,8.565000,4.320000,11.036667,0.651667,1.64000,88.200575,Satisfactory
1,"Agartala, Tripura",Agartala,Tripura,2020-11-09,42.771304,55.189130,1.668750,6.787083,9.105000,4.706667,11.220833,0.858750,6.51875,70.889445,Satisfactory
2,"Agartala, Tripura",Agartala,Tripura,2020-11-10,41.183750,54.986250,4.335000,8.927917,15.153333,7.411667,11.320000,1.085000,5.01750,68.207026,Satisfactory
3,"Agartala, Tripura",Agartala,Tripura,2020-11-11,49.401667,64.949091,1.315833,7.173333,8.950417,4.712083,11.314167,1.108750,6.50875,82.092471,Satisfactory
4,"Agartala, Tripura",Agartala,Tripura,2020-11-12,45.761250,62.839167,1.043333,7.678333,9.040417,4.743333,11.291667,0.827500,11.22000,75.941422,Satisfactory
